# Phase 4 — Feature Engineering & Preprocessing
### Social Services Demand Risk Predictor

---

This notebook transforms the cleaned datasets from Phase 3 into a **model-ready 
feature matrix**. Feature engineering is often where the most value is created 
in a data science project — better features beat better algorithms almost every time.

**Inputs:**  
- `data/processed/seifa_with_risk_tier.csv`  
- `data/processed/dss_payments_clean.csv`  
- `data/processed/master_dataset.csv`  

**Outputs:**  
- `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`  
- `data/processed/y_train.csv`, `y_val.csv`, `y_test.csv`  
- `models/preprocessing_pipeline.joblib`  

**Libraries:** pandas, numpy, scikit-learn, imbalanced-learn

## 1. Setup — Imports & Paths

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Scikit-learn — preprocessing
from sklearn.model_selection    import train_test_split, StratifiedKFold
from sklearn.preprocessing      import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.impute             import SimpleImputer
from sklearn.pipeline           import Pipeline
from sklearn.compose            import ColumnTransformer
from sklearn.feature_selection  import SelectKBest, f_classif, mutual_info_classif
from sklearn.inspection         import permutation_importance

# Class imbalance
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline      import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("  imbalanced-learn loaded successfully")
except ImportError:
    SMOTE_AVAILABLE = False
    print("  imbalanced-learn not installed — run: pip install imbalanced-learn")

# Serialisation
import joblib

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"       : 120,
    "figure.facecolor" : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

print(f"scikit-learn : imported successfully")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")

In [ ]:
# Paths

NOTEBOOK_DIR  = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR    = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR   = os.path.join(PROJECT_ROOT, "reports")

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Paths configured:")
print(f"  Processed : {PROCESSED_DIR}")
print(f"  Models    : {MODELS_DIR}")
print(f"  Reports   : {REPORTS_DIR}")

## 2. Load Data

We load the SEIFA dataset with risk tiers created in Phase 3.
This is our primary modelling dataset for the classification task.

In [ ]:
seifa_path  = os.path.join(PROCESSED_DIR, "seifa_with_risk_tier.csv")
dss_path    = os.path.join(PROCESSED_DIR, "dss_payments_clean.csv")
master_path = os.path.join(PROCESSED_DIR, "master_dataset.csv")

def load_if_exists(path, label):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  [✓] {label:<30} {df.shape[0]:>7,} rows  x  {df.shape[1]:>3} cols")
        return df
    else:
        print(f"  [✗] {label} not found — run Phase 3 first")
        return None

print("Loading datasets...")
print("-" * 55)
df_seifa  = load_if_exists(seifa_path,  "SEIFA with risk tier")
df_dss    = load_if_exists(dss_path,    "DSS payments clean")
df_master = load_if_exists(master_path, "Master dataset")
print("-" * 55)

In [ ]:
# Quick snapshot of what we're working with

if df_seifa is not None:
    print("SEIFA dataset — first 3 rows:")
    display(df_seifa.head(3))
    print()
    print("Columns:", df_seifa.columns.tolist())

## 3. Feature Catalogue

Before writing any code, we explicitly list every feature we plan to 
engineer and why. This is a professional DS practice — it keeps the 
work intentional rather than exploratory guessing.

| Feature | Source | Engineering | Rationale |
|---|---|---|---|
| `jobseeker_payment` | DSS | Raw | Direct measure of unemployment in region |
| `disability_support_pension` | DSS | Raw | Measure of disability-related disadvantage |
| `age_pension` | DSS | Raw | Indicator of aged population dependency |
| `carer_payment` | DSS | Raw | Measure of carer burden in region |
| `youth_allowance_other` | DSS | Raw | Indicator of youth unemployment |
| `parenting_payment_single` | DSS | Raw | Proxy for single parent disadvantage |
| `family_tax_benefit_a` | DSS | Raw | Measure of family financial need |
| `commonwealth_rent_assistance` | DSS | Raw | Indicator of housing affordability stress |
| `log_population` | SEIFA | log1p(usual_resident_population) | Normalises heavy right skew from large cities |
| `score_range` | SEIFA | irsd_max_sa1_score - irsd_min_sa1_score | Within-SA2 inequality proxy |
| `state_encoded` | SEIFA | Ordinal encode of state_name | Captures state-level policy differences |


## 4. Feature Engineering

In [ ]:
if df_seifa is not None:
    df = df_seifa.copy()

    # ── 4a. Log-transform population ─────────────────────────────────────
    # Population has a heavy right skew (capital cities vs tiny rural SA2s)
    # log1p(x) = log(x + 1) — handles zero values safely
    pop_col = next((c for c in df.columns if "population" in c.lower()), None)
    if pop_col:
        df[pop_col] = pd.to_numeric(df[pop_col], errors="coerce")
        df["log_population"] = np.log1p(df[pop_col])
        print(f"[✓] log_population created from '{pop_col}'")
        print(f"    Original range : {df[pop_col].min():,.0f} — {df[pop_col].max():,.0f}")
        print(f"    Log range      : {df['log_population'].min():.2f} — {df['log_population'].max():.2f}")
    else:
        df["log_population"] = np.nan
        print("[!] Population column not found — log_population set to NaN")
        print(f"    Available columns: {df.columns.tolist()}")

    print()

In [ ]:
if df_master is not None:
    # ── Score range feature ───────────────────────────────────────────
    # Difference between max and min SA1 IRSD scores within each SA2
    # captures within-region inequality — areas with high spread have
    # mixed pockets of advantage and disadvantage side by side

    min_col = next((c for c in df_master.columns if "min" in c.lower() and "score" in c.lower()), None)
    max_col = next((c for c in df_master.columns if "max" in c.lower() and "score" in c.lower()), None)

    if min_col and max_col:
        df_master[min_col] = pd.to_numeric(df_master[min_col], errors="coerce")
        df_master[max_col] = pd.to_numeric(df_master[max_col], errors="coerce")
        df_master["score_range"] = df_master[max_col] - df_master[min_col]
        print(f"[✓] score_range created  ({max_col} - {min_col})")
        print(f"    Range : {df_master['score_range'].min():.1f} — {df_master['score_range'].max():.1f}")
        print(f"    Mean  : {df_master['score_range'].mean():.1f}")
        print(f"    Nulls : {df_master['score_range'].isna().sum()}")
    else:
        df_master["score_range"] = np.nan
        print("[!] Could not find min/max SA1 score columns")
        print(f"    Available columns: {df_master.columns.tolist()}")

    print()

In [ ]:
if df_seifa is not None:
    # ── 4c. Binary disadvantage flag ─────────────────────────────────────
    # A simple binary feature: 1 = high disadvantage (decile 1-3), 0 = otherwise
    # This gives the model a strong, interpretable signal
    decile_col = next((c for c in df.columns if "irsd" in c.lower() and "decile" in c.lower()), None)

    if decile_col:
        df[decile_col] = pd.to_numeric(df[decile_col], errors="coerce")
        df["disadvantage_flag"] = (df[decile_col] <= 3).astype(int)
        n_flagged = df["disadvantage_flag"].sum()
        print(f"[✓] disadvantage_flag created from '{decile_col}'")
        print(f"    Flagged as high disadvantage : {n_flagged:,} SA2 regions")
        print(f"    Not flagged                  : {len(df) - n_flagged:,} SA2 regions")
    else:
        df["disadvantage_flag"] = np.nan
        print("[!] IRSD decile column not found")

    print()

In [ ]:
if df_seifa is not None:
    # ── 4d. Remoteness flag from SA2 name ────────────────────────────────
    # SA2 names often contain location clues: "Remote", "Rural", "CBD" etc.
    # We extract a simple 3-level remoteness proxy using regex
    import re

    sa2_name_col = next((c for c in df.columns if "name" in c.lower() and "sa2" in c.lower()), None)
    if sa2_name_col is None:
        sa2_name_col = next((c for c in df.columns if "name" in c.lower()), None)

    if sa2_name_col:
        def extract_remoteness(name):
            name = str(name).lower()
            if any(kw in name for kw in ["remote", "outback", "desert", "kimberley", "arnhem"]):
                return "Remote"
            elif any(kw in name for kw in ["rural", "regional", "country", "coast", "valley", "plains"]):
                return "Regional"
            else:
                return "Urban"

        df["remoteness_flag"] = df[sa2_name_col].apply(extract_remoteness)
        print(f"[✓] remoteness_flag created from '{sa2_name_col}'")
        print()
        for cat, count in df["remoteness_flag"].value_counts().items():
            pct = count / len(df) * 100
            print(f"    {cat:<12} {count:>5,}  ({pct:.1f}%)")
    else:
        df["remoteness_flag"] = "Unknown"
        print("[!] SA2 name column not found — remoteness_flag set to 'Unknown'")

    print()

In [ ]:
if df_seifa is not None:
    # ── 4e. Encode state column ───────────────────────────────────────────
    # State is a categorical variable — we use ordinal encoding
    # (ordered roughly by population: NSW, VIC, QLD, WA, SA, TAS, ACT, NT)
    state_col = next((c for c in df.columns if "state" in c.lower()), None)

    if state_col:
        state_order = ["New South Wales", "Victoria", "Queensland",
                       "Western Australia", "South Australia", "Tasmania",
                       "Australian Capital Territory", "Northern Territory"]

        known_states = df[state_col].dropna().unique().tolist()
        full_order   = state_order + [s for s in known_states if s not in state_order]

        state_map = {state: i for i, state in enumerate(full_order)}
        df["state_encoded"] = df[state_col].map(state_map)

        print(f"[✓] state_encoded created from '{state_col}'")
        print()
        for state, code in state_map.items():
            if state in known_states:
                n = (df[state_col] == state).sum()
                print(f"    {code}  {state:<40} {n:>5,} SA2 regions")
    else:
        df["state_encoded"] = np.nan
        print("[!] State column not found")

    print()

In [ ]:
if df_master is not None:
    # ── 4f. State encoding ────────────────────────────────────────────────
    df_master["state_encoded"] = df_master["state_name"].astype("category").cat.codes

    # ── Summary of all engineered features ───────────────────────────────
    new_features = [
        "log_population",
        "score_range",
        "state_encoded"
    ]

    print("Engineered features summary:")
    print("=" * 55)
    for feat in new_features:
        if feat in df_master.columns:
            n_null = df_master[feat].isnull().sum()
            dtype  = df_master[feat].dtype
            sample = df_master[feat].dropna().head(3).tolist()
            print(f"  {feat:<25} dtype: {str(dtype):<12} nulls: {n_null:<5} sample: {sample}")
        else:
            print(f"  {feat:<25} NOT CREATED")

In [ ]:
df_master

## 5. Define Feature Matrix & Target Variable

We now explicitly declare which columns are **features (X)** and which 
is the **target (y)**. This is the most important decision in the notebook 
— it determines what the model learns from.

In [ ]:
if df_master is not None:
    # ── Load risk_tier from saved seifa file ──────────────────────────
    seifa_risk_path = os.path.join(PROCESSED_DIR, "seifa_with_risk_tier.csv")
    if "risk_tier" not in df_master.columns:
        if os.path.exists(seifa_risk_path):
            df_seifa_risk = pd.read_csv(seifa_risk_path)[["sa2_code", "risk_tier"]]
            df_master["sa2_code"]     = df_master["sa2_code"].astype(str).str.strip()
            df_seifa_risk["sa2_code"] = df_seifa_risk["sa2_code"].astype(str).str.strip()
            df_master = df_master.merge(df_seifa_risk, on="sa2_code", how="left")
            print("[✓] risk_tier merged from seifa_with_risk_tier.csv")
        else:
            df_master["risk_tier"] = df_master["irsd_decile_aus"].apply(
                lambda d: np.nan if pd.isna(d) else
                          "High" if d <= 3 else
                          "Medium" if d <= 7 else "Low"
            )
            print("[✓] risk_tier recreated from irsd_decile_aus")
    else:
        print("[✓] risk_tier already exists in df_master")

    # ── Create remaining engineered features ──────────────────────────
    df_master["log_population"]    = np.log1p(df_master["usual_resident_population"])
    df_master["disadvantage_flag"] = (df_master["irsd_decile_aus"] <= 3).astype(int)
    df_master["state_encoded"]     = df_master["state_name"].astype("category").cat.codes
    if "score_range" not in df_master.columns:
        df_master["score_range"] = df_master["irsd_max_sa1_score"] - df_master["irsd_min_sa1_score"]
    print("[✓] log_population created")
    print("[✓] disadvantage_flag created")
    print("[✓] state_encoded created")
    print("[✓] score_range created")
    print()

    # ── Feature selection ─────────────────────────────────────────────
    # NOTE: irsd_score, irsd_decile_aus, disadvantage_flag removed —
    # they directly define risk_tier and cause data leakage (F1=1.0)
    # Features are now DSS payment patterns + population + geography
    df_master = df_master.dropna(subset=["risk_tier"])
    print(f"Rows after dropping missing risk_tier: {len(df_master):,}")
    print()

    candidate_features = []
    for col in [
        # DSS payment columns — independent predictors
        "jobseeker_payment",
        "disability_support_pension",
        "age_pension",
        "carer_payment",
        "youth_allowance_other",
        "parenting_payment_single",
        "family_tax_benefit_a",
        "commonwealth_rent_assistance",
        # Population & geography — independent of IRSD
        "log_population",
        "score_range",
        "state_encoded"
    ]:
        if col in df_master.columns:
            candidate_features.append(col)
            print(f"  [✓] Feature included : {col}")
        else:
            print(f"  [✗] Feature missing  : {col}")

    print()
    print(f"Total features selected: {len(candidate_features)}")

In [ ]:
if df_master is not None and "risk_tier" in df_master.columns:
    # Convert all feature columns to numeric — coerce any remaining strings
    for col in candidate_features:
        df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

    X = df_master[candidate_features].copy()
    y = df_master["risk_tier"].copy()

    print("Feature matrix (X) and target vector (y) defined:")
    print(f"  X shape : {X.shape}  ({X.shape[0]:,} samples, {X.shape[1]} features)")
    print(f"  y shape : {y.shape}")
    print()
    print("Target class distribution:")
    for cls, count in y.value_counts().items():
        pct = count / len(y) * 100
        bar = "█" * int(pct / 2)
        print(f"  {cls:<8} {count:>5,}  ({pct:5.1f}%)  {bar}")
    print()
    display(X.describe().T.round(3))

## 6. Handle Missing Values

Before splitting the data, we deal with any remaining missing values.
We use **median imputation** for numeric features — the median is robust 
to outliers, which is important given the skewed distributions we saw in Phase 3.

In [ ]:
if df_master is not None and "risk_tier" in df_master.columns:

    df_master["score_range"] = df_master["irsd_max_sa1_score"] - df_master["irsd_min_sa1_score"]
    print(f"[✓] score_range created — nulls: {df_master['score_range'].isnull().sum()}")
    print()

    # Rebuild X from df_master
    X = df_master[candidate_features].copy()

    print("Missing values in feature matrix before imputation:")
    print("-" * 45)
    total_missing = 0
    for col in X.columns:
        n = X[col].isnull().sum()
        total_missing += n
        pct = n / len(X) * 100
        flag = "  ← will impute" if n > 0 else ""
        print(f"  {col:<30} {n:>5,}  ({pct:.1f}%){flag}")
    print()

    if total_missing == 0:
        print("  ✓ No missing values — imputation not required")
    else:
        print(f"  Total missing cells: {total_missing:,}")
        print("  Strategy: median imputation (robust to outliers)")
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(
            imputer.fit_transform(X),
            columns=X.columns,
            index=X.index
        )
        X = X_imputed
        print()
        print("  ✓ Imputation complete — no missing values remain")

## 7. Train / Validation / Test Split

We split the data into three sets:

| Set | Size | Purpose |
|---|---|---|
| **Train** | 70% | Model learns from this |
| **Validation** | 15% | Tune hyperparameters — model never trained on this |
| **Test** | 15% | Final evaluation only — touched once at the very end |

We use `stratify=y` to ensure each split has the same class proportions 
as the full dataset. This is critical for imbalanced targets.

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    RANDOM_STATE = 42   # Fix seed for reproducibility — same split every time

    # Step 1: Split off test set (15%)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size   = 0.15,
        stratify    = y,
        random_state= RANDOM_STATE
    )

    # Step 2: Split remaining 85% into train (70%) and validation (15%)
    # 0.15 / 0.85 = 0.176 gives us 15% of the total dataset as validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size   = 0.176,
        stratify    = y_temp,
        random_state= RANDOM_STATE
    )

    total = len(X)
    print("Train / Validation / Test Split")
    print("=" * 45)
    print(f"  Total samples  : {total:,}")
    print()
    for label, X_s, y_s in [("Train", X_train, y_train),
                              ("Validation", X_val,   y_val),
                              ("Test",       X_test,  y_test)]:
        pct = len(X_s) / total * 100
        print(f"  {label:<12} {len(X_s):>5,} rows  ({pct:.1f}%)")
        for cls in sorted(y_s.unique()):
            n   = (y_s == cls).sum()
            p   = n / len(y_s) * 100
            print(f"             {cls}: {n:,}  ({p:.1f}%)")
        print()

## 8. Build the Preprocessing Pipeline

A scikit-learn `Pipeline` chains together all preprocessing steps so they 
can be applied consistently to train, validation, and test sets — and to 
new data in production. This prevents **data leakage** (accidentally fitting 
the scaler on test data) which is one of the most common errors in DS projects.

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    # All our features are numeric after encoding in earlier cells
    # so we use a single numeric pipeline

    numeric_features = X_train.columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        # StandardScaler centres each feature to mean=0, std=1
        # This ensures no feature dominates due to scale differences
        # e.g. irsd_score (range ~600-1200) vs disadvantage_flag (0 or 1)
        ("scaler",  StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features)
        ],
        remainder="drop"   # drop any columns not explicitly listed
    )

    print("Preprocessing pipeline defined:")
    print()
    print("  Step 1 — SimpleImputer(strategy='median')")
    print("           Fill any remaining NaN with the column median")
    print()
    print("  Step 2 — StandardScaler()")
    print("           Centre each feature: mean=0, std=1")
    print()
    print(f"  Applied to {len(numeric_features)} features:")
    for f in numeric_features:
        print(f"    {f}")

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    # Fit on TRAINING data only — never on validation or test
    # fit_transform learns the parameters (median, mean, std) AND transforms
    # transform only applies already-learned parameters

    X_train_scaled = preprocessor.fit_transform(X_train)
    X_val_scaled   = preprocessor.transform(X_val)
    X_test_scaled  = preprocessor.transform(X_test)

    # Wrap back into DataFrames for readability
    feature_names_out = preprocessor.get_feature_names_out()
    clean_names = [n.replace("num__", "") for n in feature_names_out]

    X_train_scaled = pd.DataFrame(X_train_scaled, columns=clean_names, index=X_train.index)
    X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=clean_names, index=X_val.index)
    X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=clean_names, index=X_test.index)

    print("Pipeline fitted and applied:")
    print(f"  X_train_scaled : {X_train_scaled.shape}")
    print(f"  X_val_scaled   : {X_val_scaled.shape}")
    print(f"  X_test_scaled  : {X_test_scaled.shape}")
    print()
    print("Scaled feature statistics (train set — should be ~mean=0, std=1):")
    display(X_train_scaled.describe().T[["mean","std","min","max"]].round(3))

## 9. Handle Class Imbalance with SMOTE

If the Phase 3 class balance check showed a ratio > 3:1 we apply SMOTE 
(Synthetic Minority Over-sampling Technique). SMOTE creates **synthetic** 
new samples for minority classes by interpolating between existing ones — 
it never duplicates, it generates new plausible data points.

**Critical rule:** SMOTE is applied to the **training set only**. 
Validation and test sets are never resampled — they must reflect the 
real-world class distribution.

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    counts    = y_train.value_counts()
    ratio     = counts.max() / counts.min()
    needs_smote = ratio > 3.0

    print("Class Balance Check — Training Set")
    print("=" * 45)
    for cls, n in counts.items():
        pct = n / len(y_train) * 100
        print(f"  {cls:<8} {n:>5,}  ({pct:.1f}%)")
    print(f"  Imbalance ratio : {ratio:.2f}:1")
    print()

    if needs_smote and SMOTE_AVAILABLE:
        print("  ⚠ Imbalance ratio > 3.0 — applying SMOTE to training set")
        smote = SMOTE(random_state=42, k_neighbors=min(5, counts.min() - 1))
        X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
        print()
        print("  After SMOTE:")
        for cls, n in pd.Series(y_train_res).value_counts().items():
            print(f"    {cls:<8} {n:>5,}")
        X_train_final = X_train_res
        y_train_final = y_train_res
    elif needs_smote and not SMOTE_AVAILABLE:
        print("  ⚠ SMOTE needed but imbalanced-learn not installed.")
        print("  Run: pip install imbalanced-learn  then restart kernel")
        print("  Falling back to class_weight='balanced' in Phase 5 models")
        X_train_final = X_train_scaled
        y_train_final = y_train
    else:
        print("  ✓ Classes balanced enough — SMOTE not required")
        print("    Use class_weight='balanced' in Phase 5 as a precaution")
        X_train_final = X_train_scaled
        y_train_final = y_train

## 10. Feature Selection

Too many features can hurt model performance through noise and overfitting. 
We use two complementary methods to score feature importance:

1. **ANOVA F-score** — tests whether the mean of each feature differs 
   significantly across risk tier classes
2. **Mutual Information** — measures non-linear statistical dependency 
   between each feature and the target

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    from sklearn.preprocessing import LabelEncoder

    # Encode target to numeric for feature selection scoring
    le           = LabelEncoder()
    y_train_enc  = le.fit_transform(y_train_final)

    # ANOVA F-score
    f_selector = SelectKBest(score_func=f_classif, k="all")
    f_selector.fit(X_train_final, y_train_enc)
    f_scores   = pd.Series(f_selector.scores_,  index=X_train_scaled.columns)

    # Mutual information
    mi_selector = SelectKBest(score_func=mutual_info_classif, k="all")
    mi_selector.fit(X_train_final, y_train_enc)
    mi_scores   = pd.Series(mi_selector.scores_, index=X_train_scaled.columns)

    # Combined feature importance DataFrame
    feat_importance = pd.DataFrame({
        "f_score"      : f_scores.round(2),
        "mutual_info"  : mi_scores.round(4),
        "f_rank"       : f_scores.rank(ascending=False).astype(int),
        "mi_rank"      : mi_scores.rank(ascending=False).astype(int),
    }).sort_values("f_score", ascending=False)

    feat_importance["avg_rank"] = (
        (feat_importance["f_rank"] + feat_importance["mi_rank"]) / 2
    ).round(1)

    print("Feature Importance Rankings")
    print("=" * 65)
    display(feat_importance)

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # F-score chart
    f_sorted = feat_importance["f_score"].sort_values()
    axes[0].barh(f_sorted.index, f_sorted.values,
                 color="#1D9E75", edgecolor="white", linewidth=0.5)
    axes[0].set_title("ANOVA F-Score per Feature\n(Higher = more discriminative)",
                      fontweight="bold")
    axes[0].set_xlabel("F-Score")

    # Mutual Information chart
    mi_sorted = feat_importance["mutual_info"].sort_values()
    axes[1].barh(mi_sorted.index, mi_sorted.values,
                 color="#534AB7", edgecolor="white", linewidth=0.5)
    axes[1].set_title("Mutual Information per Feature\n(Higher = more informative)",
                      fontweight="bold")
    axes[1].set_xlabel("Mutual Information Score")

    plt.suptitle("Feature Importance — Two Methods Compared",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "11_feature_importance.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/11_feature_importance.png")

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    # Select features that rank in the top half by average rank
    threshold    = feat_importance["avg_rank"].median()
    selected_features = feat_importance[
        feat_importance["avg_rank"] <= threshold
    ].index.tolist()

    print(f"Feature selection threshold (avg_rank <= {threshold:.1f}):")
    print()
    print("  SELECTED features:")
    for f in selected_features:
        row = feat_importance.loc[f]
        print(f"    {f:<30} avg_rank: {row['avg_rank']}")
    print()
    dropped = [f for f in feat_importance.index if f not in selected_features]
    print("  DROPPED features (low importance):")
    for f in dropped:
        row = feat_importance.loc[f]
        print(f"    {f:<30} avg_rank: {row['avg_rank']}")

    # Apply selection to all splits
    X_train_sel = X_train_final[selected_features] if hasattr(X_train_final, "__getitem__") else X_train_final
    X_val_sel   = X_val_scaled[selected_features]
    X_test_sel  = X_test_scaled[selected_features]

    print()
    print(f"Final feature matrix shape  : {X_train_sel.shape}")

## 11. Visualise Final Feature Distributions

A final sanity check — we plot the distribution of each selected feature 
split by risk tier to confirm the features genuinely separate the classes.

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:
    # Combine train features + labels for plotting
    plot_df = X_train_scaled[selected_features].copy()
    plot_df["risk_tier"] = y_train.values[:len(plot_df)] if len(y_train) == len(plot_df) else y_train.reset_index(drop=True)

    tier_palette = {"High": "#993C1D", "Medium": "#BA7517", "Low": "#1D9E75"}

    n_feats = len(selected_features)
    ncols   = min(3, n_feats)
    nrows   = (n_feats + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, feat in enumerate(selected_features):
        for tier, color in tier_palette.items():
            data = plot_df[plot_df["risk_tier"] == tier][feat].dropna()
            if len(data) > 0:
                axes[i].hist(data, bins=25, alpha=0.55, color=color,
                             label=tier, edgecolor="none")
        axes[i].set_title(feat.replace("_", " ").title(), fontweight="bold")
        axes[i].set_xlabel("Scaled value")
        axes[i].set_ylabel("Count")
        axes[i].legend(fontsize=8)

    # Hide any empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Selected Feature Distributions by Risk Tier\n(Good separation = features are useful)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "12_feature_distributions_by_tier.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/12_feature_distributions_by_tier.png")

## 12. Save All Splits & Pipeline to Disk

In [ ]:
if df_seifa is not None and "risk_tier" in df.columns:

    saves = [
        (X_train_sel,  "X_train.csv"),
        (X_val_sel,    "X_val.csv"),
        (X_test_sel,   "X_test.csv"),
    ]

    # y splits — save as DataFrames
    for y_split, fname in [(y_train_final, "y_train.csv"),
                            (y_val,         "y_val.csv"),
                            (y_test,        "y_test.csv")]:
        out = os.path.join(PROCESSED_DIR, fname)
        pd.Series(y_split).to_csv(out, index=False, header=["risk_tier"])
        size = os.path.getsize(out) / 1024
        print(f"  [✓] {fname:<20} {size:>8.1f} KB")

    # X splits
    for X_split, fname in saves:
        out = os.path.join(PROCESSED_DIR, fname)
        if hasattr(X_split, "to_csv"):
            X_split.to_csv(out, index=False)
        else:
            pd.DataFrame(X_split, columns=selected_features).to_csv(out, index=False)
        size = os.path.getsize(out) / 1024
        print(f"  [✓] {fname:<20} {size:>8.1f} KB")

    # Save preprocessing pipeline
    pipeline_path = os.path.join(MODELS_DIR, "preprocessing_pipeline.joblib")
    joblib.dump(preprocessor, pipeline_path)
    size = os.path.getsize(pipeline_path) / 1024
    print(f"  [✓] preprocessing_pipeline.joblib  {size:>8.1f} KB")

    # Save selected feature list
    feat_path = os.path.join(PROCESSED_DIR, "selected_features.txt")
    with open(feat_path, "w") as f:
        f.write("\n".join(selected_features))
    print(f"  [✓] selected_features.txt")

    print()
    print("All splits and pipeline saved successfully.")

## 13. Phase 4 Summary

In [ ]:
print("=" * 60)
print("  PHASE 4 COMPLETE — Feature Engineering & Preprocessing")
print("=" * 60)
print()

steps = [
    ("Log-transform population",   "Stabilised right-skewed population column"),
    ("Score range feature",        "Engineered spread between IRSAD and IRSD"),
    ("Disadvantage flag",          "Binary: IRSD decile <= 3"),
    ("Remoteness flag",            "Extracted Urban / Regional / Remote from SA2 name"),
    ("State encoding",             "Ordinal encoded — 8 Australian states/territories"),
    ("Imputation",                 "Median imputation for any missing values"),
    ("Standard scaling",           "Mean=0, Std=1 applied to all numeric features"),
    ("Train/val/test split",       "70% / 15% / 15% with stratification"),
    ("SMOTE",                      "Applied if class imbalance ratio > 3:1"),
    ("Feature selection",          "ANOVA F-score + Mutual Information ranking"),
    ("Pipeline serialised",        "models/preprocessing_pipeline.joblib"),
]

for step, detail in steps:
    print(f"  ✓  {step:<35} {detail}")

print()
print("  Output files:")
for fname in ["X_train.csv", "X_val.csv", "X_test.csv",
              "y_train.csv", "y_val.csv", "y_test.csv",
              "selected_features.txt"]:
    print(f"    data/processed/{fname}")
print("    models/preprocessing_pipeline.joblib")
print()
print("  Next notebook: 05_modelling.ipynb")
print("=" * 60)

## 14. Phase 4 Complete

**What this notebook produced:**

- ✓ 5 engineered features added to the dataset
- ✓ Missing values handled with median imputation
- ✓ Train / Validation / Test split (70/15/15) with stratification
- ✓ StandardScaler pipeline — fitted on train, applied to all splits
- ✓ SMOTE applied if class imbalance detected
- ✓ Feature importance ranked by ANOVA F-score and Mutual Information
- ✓ Low-importance features dropped
- ✓ 2 charts saved to `reports/`
- ✓ All splits saved to `data/processed/`
- ✓ Preprocessing pipeline saved to `models/`

**Next:** `05_modelling.ipynb` — train Logistic Regression → Random Forest → 
XGBoost, tune hyperparameters, evaluate with F1/ROC-AUC, and generate SHAP 
explainability plots.